# Learning about Reinforcement Learning #

<a href="https://colab.research.google.com/github/deepmind/educational/blob/master/colabs/introductory/reinforcement_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



> <p><small><small><b>Copyright 2020 DeepMind Technologies Limited.</b></p>
> <p><small><small> Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at </p>
> <p><small><small> <a href="https://www.apache.org/licenses/LICENSE-2.0">https://www.apache.org/licenses/LICENSE-2.0</a> </p>
> <p><small><small> Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License. </p>



**Aim**

Reinforcement learning has been successful in solving challenging problems, such as the games of Go and StarCraft. It has significant potential for many real-world problems, such as drug discovery, global warming, or even discovering new scientific theories in physics, mathematics, and other fundamental sciences.


This tutorial will go through an example of applying Reinforcement Learning to solve a problem and familiarise you with basic concepts used to formalise it. We will load an environment and an agent, specify a reward function, and then train a neural network to perform the task!

**Disclaimer**

This code is intended for educational purposes, and in the name of readability for a non-technical audience does not always follow best practices for software engineering.


**Links to resources**
- [What is Colab?](https://colab.sandbox.google.com/notebooks/intro.ipynb) If you have never used Colab before, get started here!

## Reinforcement Learning


In Reinforcement Learning we want to train an **agent** to maximise the total **reward** it receives within a fixed duration of interacting with an **environment**.

The following diagram illustrates the interaction between the agent and environment.
<center>
<img src="https://storage.googleapis.com/dm-educational/assets/reinforcement_learning/rl_loop_illustrated.png" width="500" />
</center>

In this tutorial we will focus on simple environments to familiarise you with the process of training an agent. The simplest environment, shown below, is CartPole. This environment consists of a pole attached to a cart via a hinge. The agent needs to move the cart to the left or to the right in order to prevent the pole from falling over - this can be learned in just a few minutes.

<center>
<img src="https://storage.googleapis.com/dm-educational/assets/reinforcement_learning/42135683-dde5c6f0-7d13-11e8-90b1-8770df3e40cf.gif" width="500" />
</center>



#Pre-requisites#

Before we start, we'll have to set up a few things. This colab will involve running a number of *cells*, each containing some code. If you look at the cell below, and hover over the brackets to the top left, it should turn into a play sign, that can be used to start or stop running the code in the cell.

Click the play button on the next three cells below to install the software packages, import Python modules, and implement some functions that we'll use under the hood. (You should see it change to a "stop" icon while it's running.)

This should only take around 30 seconds.

In [ ]:
#@title Install software packages for Colab {'form-width':'30%'}
import sys
import subprocess

packages = [
    'gymnasium[classic-control]',
    'imageio',
    'matplotlib',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Colab packages are ready.')


In [ ]:
#@title Import python libraries {'form-width':'30%'}

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import base64
import collections
import os
import random

os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')

import gymnasium as gym
import imageio.v2 as imageio
import IPython
import matplotlib.pyplot as plt
import numpy as np

import warnings
warnings.filterwarnings('ignore')

plt.rcdefaults()

np.random.seed(0)
random.seed(0)


In [ ]:
#@title Set up utilities {'form-width':'30%'}


class TimeStep(object):
  """Small dm_env-like timestep used by this notebook."""

  def __init__(self, observation, reward=0.0, done=False):
    self.observation = np.asarray(observation, dtype=np.float32)
    self.reward = float(reward)
    self._done = bool(done)

  def last(self):
    return self._done


class GymCartPoleWrapper(object):
  """Adapter that gives Gymnasium CartPole the API used below."""

  def __init__(self, seed=0):
    self.env = gym.make('CartPole-v1', max_episode_steps=200, render_mode='rgb_array')
    self.action_space = self.env.action_space
    self.observation_space = self.env.observation_space
    self.seed = seed
    self.episode_index = 0

  def reset(self):
    observation, _ = self.env.reset(seed=self.seed + self.episode_index)
    self.episode_index += 1
    return TimeStep(observation)

  def step(self, action):
    observation, reward, terminated, truncated, _ = self.env.step(int(action))
    return TimeStep(observation, reward, terminated or truncated)

  def render(self, mode='rgb_array'):
    return self.env.render()


def step_agent_in_environment(env, agent=None, num_episodes=3):
  """Steps an agent in an environment."""
  frames = []
  actions = []

  for n in range(num_episodes):
    timestep = env.reset()
    while not timestep.last():
      frames.append(env.render(mode='rgb_array'))
      if callable(agent):
        action = agent(timestep.observation)
      else:
        action = agent.select_action(timestep.observation, training=False)
      actions.append(action)
      timestep = env.step(action)

  return frames, actions


def show_video(frames):
  """Show an animated GIF inline."""
  if not frames:
    return IPython.display.HTML('<p>No frames to display.</p>')

  video_filename = 'cartpole.gif'
  imageio.mimsave(video_filename, frames, duration=1 / 30)
  gif = open(video_filename, 'rb').read()
  b64 = base64.b64encode(gif).decode()
  tag = '<img src="data:image/gif;base64,{0}" width="640" height="480" />'.format(b64)
  return IPython.display.HTML(tag)


print('All set!')


## Environments

Next, let's load the RL environment we want to use!

Environments in RL represent the task or problem that we are trying to solve. There are many types of environments, such as computer games, simulated robotics settings, etc. Some of these can take several hours or days for an agent to learn!

First pick [CartPole](https://github.com/openai/gym/wiki/CartPole-v0) from the drop down menu as it is the fastest environment to train on. Later you can choose another environment to play with!

Running this cell will show you an image of what the environment looks like.

In [ ]:
#@title Load an environment
environment_name = 'CartPole'  #@param ['CartPole']

if 'CartPole' in environment_name:
  environment_train = GymCartPoleWrapper(seed=0)
  environment = GymCartPoleWrapper(seed=10_000)
else:
  raise ValueError('Unknown environment: {}.'.format(environment_name))

print('Loaded environment:', environment_name)
print('Observation size:', environment.observation_space.shape[0])
print('Number of actions:', environment.action_space.n)


# Random Agent

Before we train an agent, we will start with a *random agent*. The random agent does not take environment observations into account when choosing to take an action and just chooses actions randomly from the set of all possible actions. You can generate random actions using the code below and see the behaviour of the agent:


In [ ]:
action_space = environment.action_space

def int_random_action(state):
  # state is unused for random agent
  return action_space.sample()


output = environment.reset()
print('random action:', int_random_action(None))
print('random action:', int_random_action(None))
print('random action:', int_random_action(None))
print('random action:', int_random_action(None))

frames, actions = step_agent_in_environment(
    env=environment, agent=int_random_action, num_episodes=5)

print('actions = {}'.format(actions))

show_video(frames)


# Custom Agent

<center>
<img src="https://storage.googleapis.com/dm-educational/assets/reinforcement_learning/random_policy.png" width="500" />
</center>

Let's now try and manually code a better behaviour into the agent. The code below defines a way to run a custom set of actions. A value of 0 for the action means the cart is pushed left, and a value of 1 means the cart is pushed right.
Currently, the code is set to a constant zero, which pushes the cart left all the time (you'll see this if you run it and play the video!).

You can modify the function to specify what action should be taken depending on the observation.
One thing you could try is to use an ***if statement***, which in Python, looks like the following.

```
if [some_condition]:
  action = [some_value]
else:
  action = [some_other_value]
```
Looking at the code below, you have access to the `cart_position`, `cart_velocity`, `pole_angle`, and `pole_velocity_at_tip`.
Can you think of what conditions you might use to set the actions based on these?
**Ask for help if you're not sure!**

In [ ]:
# Example parameter set for a simple linear policy.
# Change these values if the exam gives different conditions.
POSITION_WEIGHT = 0.05
CART_VELOCITY_WEIGHT = 0.1
ANGLE_WEIGHT = 1.0
POLE_VELOCITY_WEIGHT = 0.5
DECISION_THRESHOLD = 0.0


def custom_action_for_cartpole(state):
  # For CartPole only.
  cart_position = state[0]
  cart_velocity = state[1]
  pole_angle = state[2]
  pole_velocity_at_tip = state[3]

  # Combine all observation features into one decision score.
  # Positive score means push right, negative score means push left.
  score = (
      POSITION_WEIGHT * cart_position +
      CART_VELOCITY_WEIGHT * cart_velocity +
      ANGLE_WEIGHT * pole_angle +
      POLE_VELOCITY_WEIGHT * pole_velocity_at_tip
  )

  if score < DECISION_THRESHOLD:
    action = 0  # Push left.
  else:
    action = 1  # Push right.

  return action


# One worked example with a made-up state.
example_state = np.array([0.10, -0.20, 0.04, 0.30], dtype=np.float32)
example_score = (
    POSITION_WEIGHT * example_state[0] +
    CART_VELOCITY_WEIGHT * example_state[1] +
    ANGLE_WEIGHT * example_state[2] +
    POLE_VELOCITY_WEIGHT * example_state[3]
)
example_action = custom_action_for_cartpole(example_state)

print('Example state:', example_state)
print('Example score:', example_score)
print('Example action:', example_action)


output = environment.reset()

frames, actions = step_agent_in_environment(
    env=environment, agent=custom_action_for_cartpole, num_episodes=5)

show_video(frames)


### Exam notes: custom agent

The CartPole state has four values: cart position, cart velocity, pole angle, and pole angular velocity at the tip. The custom agent above is a parameterized linear policy: it multiplies each state feature by a tunable weight, sums the result into one score, and then chooses left or right depending on a threshold.

This is not learning yet: the rule is written by hand and does not improve from experience. It is useful as a readable baseline before training a DQN agent.


# How to train your agent? #

Getting an agent to solve a task by using random actions is clearly not optimal.

There are a lot of solutions invented by scientists around the world to solve the reinforcement learning problem. They all have different strengths and weaknesses. One of the most famous modern methods is called **Deep Q Networks (DQN)**, proposed by scientists at DeepMind in 2014, and was the first RL method to solve Atari games from images.

Here we will use this method to train our agent. You don't need to worry about all the details but if you are interested you can later look at how changing some aspects of the code makes your agent learn faster or slower.

## Set up the agent
Hit run in the next cell to set up the agent. This local version uses a small NumPy Q-network, an optimizer implemented by gradient descent, and a replay buffer for stored transitions.


In [ ]:
#@title Agent setup  {'form-width':'30%'}

Transition = collections.namedtuple(
    'Transition', ['state', 'action', 'reward', 'next_state', 'done'])


class DQNAgent(object):
  """A small NumPy DQN-style agent with a replay buffer."""

  def __init__(
      self,
      state_size,
      num_actions,
      learning_rate,
      batch_size=64,
      max_replay_size=10000,
      hidden_size=64,
      gamma=0.99,
      epsilon_start=1.0,
      epsilon_end=0.05,
      epsilon_decay_steps=5000,
      seed=0,
  ):
    self.state_size = state_size
    self.num_actions = num_actions
    self.learning_rate = learning_rate
    self.batch_size = batch_size
    self.max_replay_size = max_replay_size
    self.gamma = gamma
    self.epsilon_start = epsilon_start
    self.epsilon_end = epsilon_end
    self.epsilon_decay_steps = epsilon_decay_steps
    self.rng = np.random.default_rng(seed)
    self.replay = []
    self.replay_index = 0
    self.training_steps = 0

    self.W1 = self.rng.normal(0.0, 0.1, size=(state_size, hidden_size))
    self.b1 = np.zeros(hidden_size)
    self.W2 = self.rng.normal(0.0, 0.1, size=(hidden_size, num_actions))
    self.b2 = np.zeros(num_actions)

  def _features(self, states):
    states = np.asarray(states, dtype=np.float32)
    return np.clip(states, -5.0, 5.0)

  def _forward(self, states):
    x = self._features(states)
    z1 = x @ self.W1 + self.b1
    h1 = np.maximum(z1, 0.0)
    q_values = h1 @ self.W2 + self.b2
    return x, z1, h1, q_values

  def _epsilon(self):
    fraction = min(1.0, self.training_steps / self.epsilon_decay_steps)
    return self.epsilon_start + fraction * (self.epsilon_end - self.epsilon_start)

  def select_action(self, state, training=True):
    if training and self.rng.random() < self._epsilon():
      action = self.rng.integers(self.num_actions)
    else:
      _, _, _, q_values = self._forward(np.asarray(state)[None, :])
      action = int(np.argmax(q_values[0]))

    if training:
      self.training_steps += 1
    return int(action)

  def observe(self, state, action, reward, next_state, done):
    transition = Transition(
        np.asarray(state, dtype=np.float32),
        int(action),
        float(reward),
        np.asarray(next_state, dtype=np.float32),
        bool(done),
    )
    if len(self.replay) < self.max_replay_size:
      self.replay.append(transition)
    else:
      self.replay[self.replay_index] = transition
      self.replay_index = (self.replay_index + 1) % self.max_replay_size

  def update(self):
    if len(self.replay) < self.batch_size:
      return None

    indices = self.rng.choice(len(self.replay), size=self.batch_size, replace=False)
    batch = [self.replay[i] for i in indices]

    states = np.stack([t.state for t in batch])
    actions = np.asarray([t.action for t in batch])
    rewards = np.asarray([t.reward for t in batch])
    next_states = np.stack([t.next_state for t in batch])
    dones = np.asarray([t.done for t in batch], dtype=np.float32)

    x, z1, h1, q_values = self._forward(states)
    _, _, _, next_q_values = self._forward(next_states)
    targets = rewards + self.gamma * (1.0 - dones) * np.max(next_q_values, axis=1)

    chosen_q_values = q_values[np.arange(self.batch_size), actions]
    errors = chosen_q_values - targets
    loss = float(np.mean(errors ** 2))

    d_q = np.zeros_like(q_values)
    d_q[np.arange(self.batch_size), actions] = 2.0 * errors / self.batch_size

    dW2 = h1.T @ d_q
    db2 = np.sum(d_q, axis=0)
    dh1 = d_q @ self.W2.T
    dz1 = dh1 * (z1 > 0.0)
    dW1 = x.T @ dz1
    db1 = np.sum(dz1, axis=0)

    gradients = [dW1, db1, dW2, db2]
    for gradient in gradients:
      np.clip(gradient, -1.0, 1.0, out=gradient)

    self.W1 -= self.learning_rate * dW1
    self.b1 -= self.learning_rate * db1
    self.W2 -= self.learning_rate * dW2
    self.b2 -= self.learning_rate * db2

    return loss


def setup_agent(
    environment,
    learning_rate,
    batch_size=64,
    max_replay_size=1000,
    logger=None,
):
  """Set up the agent before training."""
  del logger
  return DQNAgent(
      state_size=environment.observation_space.shape[0],
      num_actions=environment.action_space.n,
      learning_rate=learning_rate,
      batch_size=batch_size,
      max_replay_size=max_replay_size,
      seed=0,
  )


## How to evaluate success?#
<center>
<img src="https://storage.googleapis.com/dm-educational/assets/reinforcement_learning/rl_question.png" width="200" />
</center>


Before we start training, how can we tell if our agents are any good? To answer that, we need some measure of 'goodness', which is usually related to the total rewards obtained over an episode, which is called the **return**. We can also talk about **loss** instead of reward: these are effectively just the opposite of one another. So we want to maximise the reward or minimise the loss.

For example in CartPole the agent gets a positive reward of $1$ per timestep. The episode ends if the agent drops the pole or the cart is far from the centre. So in order to get as much reward as possible, the agent just needs to make the episode last as long as possible!



## Set up training
Hit run in the next cell to set up the training loop. It repeatedly collects transitions from the environment, stores them in replay memory, and updates the Q-network from random batches.


In [ ]:
#@title Training loop {'form-width':'30%'}

def train(environment, agent, num_training_episodes, log_every=10):
  """Train the agent with replay-buffer Q-learning."""
  min_actor_steps_before_learning = 1000
  num_learner_steps_per_iteration = 1
  all_returns = []
  recent_losses = []

  learner_steps_taken = 0
  actor_steps_taken = 0
  for episode in range(num_training_episodes):
    timestep = environment.reset()
    episode_return = 0.0

    while not timestep.last():
      action = agent.select_action(timestep.observation, training=True)
      next_timestep = environment.step(action)

      agent.observe(
          state=timestep.observation,
          action=action,
          reward=next_timestep.reward,
          next_state=next_timestep.observation,
          done=next_timestep.last(),
      )

      episode_return += next_timestep.reward
      actor_steps_taken += 1
      timestep = next_timestep

      if actor_steps_taken >= min_actor_steps_before_learning:
        for learner_step in range(num_learner_steps_per_iteration):
          loss = agent.update()
          if loss is not None:
            recent_losses.append(loss)
            learner_steps_taken += 1

    if episode % log_every == 0 or episode == num_training_episodes - 1:
      mean_loss = np.mean(recent_losses[-100:]) if recent_losses else float('nan')
      print(f'Episode: {episode} | Return: {episode_return:.0f} | '
            f'Learner steps: {learner_steps_taken} | '
            f'Actor steps: {actor_steps_taken} | '
            f'Mean loss: {mean_loss:.4f}')
    all_returns.append(episode_return)

  return all_returns


## Train the agent!
We need to specify the `hyperparameters` of our experiment to start the training. These include how long the agent trains, how fast it learns, and how much replay memory it keeps.


In [ ]:
#@title Train the agent, using some specific hyperparameters

num_training_episodes = 200  # @param {type:"integer"}
learning_rate = 3e-4  # @param {type:"number"}

# Other parameters
batch_size = 64
max_replay_size = 100000

# Set how often to print logs
log_every = 10

# Setup the agent
agent = setup_agent(
    environment_train,
    learning_rate,
    batch_size=batch_size,
    max_replay_size=max_replay_size)

# Use the training environment to train the agent
returns = train(environment_train, agent, num_training_episodes, log_every)

print('Final 10-episode mean return:', np.mean(returns[-10:]))


In [ ]:
#@title Plot the training curve {'form-width':'30%'}

plt.figure(figsize=(10, 5))
plt.plot(range(0, num_training_episodes), returns)
plt.grid(True)
plt.xlabel('Episodes', fontsize=15)
plt.ylabel('Total reward', fontsize=15)
plt.tick_params(labelsize=15)
plt.locator_params(nbins=10)
plt.show()


In [ ]:
#@title Show video of the trained agent's behaviour {'form-width':'30%'}

frames, actions = step_agent_in_environment(
    env=environment, agent=agent, num_episodes=5)

print('Evaluation episode count:', 5)
print('Evaluation frame count:', len(frames))
print('First 20 evaluation actions:', actions[:20])
show_video(frames)


## Analysis and post-mortem (what went wrong?)

In this environment, by the end of training the agent should be able to reach a maximum reward of $200$ in an episode. Based on the plot above, and the printouts during training, did it get there?

If not, perhaps we can try training again with some different hyperparameters?
Try going back and increasing the number of training steps to $400$, which means the agent will learn for longer.
You could also (separately) try increasing the *learning rate* from 3e-4 (i.e. $3 \times 10^{-4}$) to 1e-3 (i.e. $1 \times 10^{-3}$): this quantity varies how much the agent changes on each training step.

### Exam notes: what happens during training

DQN trains a neural network to approximate the Q-function: the expected future reward for each state-action pair. The agent selects an action, steps the environment, receives the next state and reward, stores the transition in a replay buffer, and updates the network from random batches of stored transitions. The replay buffer makes training more stable because consecutive environment steps are highly correlated.

In CartPole, the reward is 1 for every timestep while the pole has not fallen and the cart has not moved too far away. Therefore, the episode return is equal to the episode length. For `CartPole-v0`, the maximum return is 200, so a curve close to 200 means the agent keeps the pole balanced for almost the full episode.

Hyperparameters matter: more episodes provide more experience, the learning rate controls the size of neural network weight updates, the batch size controls how many replay samples are used per update, and the maximum replay size limits the agent memory.


## Additional activities ##
Now that you've used reinforcement learning to train an agent to perform the CartPole task, try giving the task a go yourself and see how you do! You can try controlling CartPole via keyboard at this link: https://fluxml.ai/experiments/cartPole/

If you have any spare time, go back and change the hyperparameters to make your agent learn faster! You can also choose a different environment (eg. Atari or MountainCar), and run everything again!

Note that Atari might take quite a long time to train - see for yourself!



